# Phase 3 — Data Preprocessing & Feature Engineering Pipeline


In [ ]:
# Phase 3 — Data Preprocessing & Feature Engineering Pipeline

# ============================================================
# Phase 3 — Setup and constants
# ============================================================
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_FILE = Path("/kaggle/input/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv")

SENSOR_COLUMNS = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]
BINARY_TARGET = "Machine failure"
FAILURE_TYPE_COLUMNS = ["TWF", "HDF", "PWF", "OSF", "RNF"]
MULTI_CLASS_TARGET = "Failure Type"
CATEGORICAL_COL = "Type"
ID_COLUMNS = ["UDI", "Product ID"]

# Load data
df = pd.read_csv(DATA_FILE)

# Build multi-class target (same policy as Phase 1)
flag_sum = df[FAILURE_TYPE_COLUMNS].sum(axis=1)
df[MULTI_CLASS_TARGET] = "No Failure"
failed_mask = df[BINARY_TARGET] == 1
df.loc[failed_mask, MULTI_CLASS_TARGET] = (
    df.loc[failed_mask, FAILURE_TYPE_COLUMNS]
    .apply(lambda row: "+".join(row.index[row == 1]) or "Unknown", axis=1)
)

print(f"✅ Data ready: {df.shape[0]:,} rows × {df.shape[1]} columns")

# ============================================================
# Feature engineering
# ============================================================
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create new physical features derived from known failure rules.
    - Power [W]     : power = torque × speed (rad/s)
    - Temp Diff [K] : process temp - air temp
    - Overstrain    : tool wear × torque
    """
    df = df.copy()

    speed_rad_s = df["Rotational speed [rpm]"] * (2 * np.pi / 60)
    df["Power [W]"] = df["Torque [Nm]"] * speed_rad_s

    df["Temp Diff [K]"] = (
        df["Process temperature [K]"] - df["Air temperature [K]"]
    )

    df["Overstrain [min·Nm]"] = df["Tool wear [min]"] * df["Torque [Nm]"]

    return df

df = engineer_features(df)
ENGINEERED_COLUMNS = ["Power [W]", "Temp Diff [K]", "Overstrain [min·Nm]"]
print("New features:")
print(df[ENGINEERED_COLUMNS].describe().round(2).to_string())

# ============================================================
# Prepare features and target
# ============================================================
feature_cols = SENSOR_COLUMNS + ENGINEERED_COLUMNS + [CATEGORICAL_COL]
X = df[feature_cols]
y = df[BINARY_TARGET]

numeric_cols = [c for c in X.columns if X[c].dtype in ["int64", "float64"]]
categorical_cols = [c for c in X.columns if X[c].dtype == "object"]

print(f"Numeric columns ({len(numeric_cols)}):")
print(numeric_cols)
print(f"\nCategorical columns ({len(categorical_cols)}):")
print(categorical_cols)
print(f"\nX shape: {X.shape}, y shape: {y.shape}")

# ============================================================
# Preprocessing pipeline
# ============================================================
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=[["L", "M", "H"]])),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

print("✅ Preprocessor built")

# ============================================================
# Stratified split
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y,
)

print("Target distribution in train:")
print(y_train.value_counts().to_string())
print(f"Failure rate in train: {y_train.mean() * 100:.2f}%")

print("\nTarget distribution in test:")
print(y_test.value_counts().to_string())
print(f"Failure rate in test: {y_test.mean() * 100:.2f}%")

preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\n✅ Shapes after preprocessing:")
print(f"   Train: {X_train_processed.shape}")
print(f"   Test : {X_test_processed.shape}")

# ============================================================
# Imbalanced data handling
# ============================================================
n_failures = (y_train == 1).sum()
n_normal   = (y_train == 0).sum()
scale_pos_weight = n_normal / n_failures

print(f"Normal samples in train : {n_normal}")
print(f"Failure samples in train: {n_failures}")
print(f"scale_pos_weight        : {scale_pos_weight:.2f}")
